# Fee Chart Code

- OS meaning operating system
- GetCWD meaning Working Directory
- Telling us where Pythons pulling files from

In [2]:
import os
os.getcwd(), os.listdir()

('/home/9a8ae02a-2a7d-4aaa-a79e-235075eedfa6/Fee Chart Code',
 ['staffdierectory.csv',
  '2007161.00project_hours_summary.csv',
  'FeeChartCode-Copy1.ipynb',
  'BR_241500.00.csv',
  'project_hours_summary.csv',
  '.ipynb_checkpoints',
  'FeeChartCode.ipynb',
  '2007161.00_hours_summary.csv',
  'Untitled3.ipynb',
  'project_hours_summary.xlsx',
  'BR_200761.01.csv',
  'feechartcharacterseparation.ipynb',
  'BR_221384.00.csv'])

In [68]:
os.listdir("/home/9a8ae02a-2a7d-4aaa-a79e-235075eedfa6/Fee Chart Code")
path = "BR_211338.01.csv"   # ← example, USE EXACT NAME YOU SEE

In [69]:
import pandas as pd

data = pd.read_csv("BR_211338.01.csv", encoding="cp1252")

- Pandas is main package used for tables of data
- Insert file path for csv
- Import csv file and turn it into a table
- Cp1252 reads the file in Windows terms instead of Excels

In [70]:
data.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,Project Detail,NaN,NaN,NaN,NaN,NaN,NaN,"Tuesday, January 27, 2026",NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3:59:16 PM,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


- Open: open file so I can see inside it
- "r" meaning read the file
- Errors="ignore" meaning if there files that Python can't interpret, skip them

In [71]:
with open(path, "r", encoding="cp1252", errors="ignore") as f:
    lines = [ln.strip() for ln in f.readlines()]

lines[:20] 

[',,,,,,,,',
 'Project Detail,,,,,,,"Tuesday, January 27, 2026",',
 ',,,,,,,,',
 ',,,,,,,3:59:16 PM,',
 ',,,,,,,,',
 ',,,,,,,,',
 ',"JEO Consulting Group, Inc.",,,,,,,',
 ',,Job-to-Date through 1/31/2026,,,,,,',
 ',,,,,,,,',
 ',,,,,,,,',
 'Estimate Overhead,,,"Total',
 'Hours",Billing,,,,',
 'Project Number: R211338.01 Callaway New Pool Design and Constructio,,,,,,,,',
 'Phase Number: 300FD Final Design (Architecture),,,,,,,,',
 '"Task Number: 001 Design, Meetings, Corrections, Specifica",,,,,,,,',
 'Labor,,,,,,,,',
 '"00039   F      10199  Divis, Sarah    2/14/2023 ",,,.25,27.50,,,,',
 '"00039   F      10251  Bellamy, Adam    1/18/2023 ",,,1.50,247.50,,,,',
 'Kickoff design meeting,,,,,,,,',
 '"00039   F      10251  Bellamy, Adam    1/19/2023 ",,,1.00,165.00,,,,']

In [72]:
# import Pythons regular expression module to search for patterns
import re
phase = None
rows = []

time_re  = re.compile(r"([A-Za-z'\-]+,\s*[A-Za-z'\-]+)\s+(\d{1,2}/\d{1,2}/\d{4}).*?(\d+\.\d+|\.\d+)")
phase_re = re.compile(r"Phase Number:\s*(.*)")

- Re.compile: creates a search pattern for Python in Excel
- "[A-Za-z'\-]+,\s*[A-Za-z'\-]+)": last name, first name
- (\d{1,2}/\d{1,2}/\d{4})): date
- (\d+\.\d+|\.\d+)): Time in hours
- VERBOSE: Allows readable formatting
- (.*): Captures any phase number
- .search: looks for phase numebr anywhere in the line
- .srip(): removes extra spaces
- float(): converts 0.25 to .25

In [73]:
# develop pattern to find the name and hours of time using _re for search
name_date_re = re.compile(r"([A-Za-z'\-]+,\s*[A-Za-z'\-]+)\s+(\d{1,2}/\d{1,2}/\d{4})")
num_re = re.compile(r"[-+]?\d*\.?\d+")

# phase will update as phython sifts through data
rows = []
phase = None

# sift through the report using enumerate
for i, ln in enumerate(lines):
# detect and store the current phase
    mphase = phase_re.search(ln)
    if mphase:
        phase = mphase.group(1).replace(",", " ").strip()
        phase = " ".join(phase.split())
        continue
# detect line that could have time entry (if line doesnt contain name and date, ignore)
    md = name_date_re.search(ln)
    if not md:
        continue
# extract the name and date by groups()
    name, date = md.groups()

# take everything AFTER the date and extract numbers
    after_date = ln.split(date, 1)[1]
    nums = [float(x) for x in num_re.findall(after_date)]

    # choose the number that looks like "hours"
    hours_candidates = [x for x in nums if -24 <= x <= 24]
    if not hours_candidates:
        continue
# pull the first number from that row
    hours = hours_candidates[0]  

# look one line below the index and use replace to get ride of extra characters
    j = i + 1
    def clean_line(s): return s.replace(",", "").replace('"', "").strip()
    while j < len(lines) and clean_line(lines[j]) == "":
        j += 1
    task = clean_line(lines[j]) if j < len(lines) else ""
# save one clean record of compiled data (compd)
    rows.append({"Phase": phase, "Name": name.strip(), "Date": date.strip(), "Hours": hours, "Task": task})

compd = pd.DataFrame(rows)

In [74]:
# convert hour values to numeric values
def to_float_hours(s):
    s = s.strip()
    if s.startswith("(") and s.endswith(")"):
        return -float(s[1:-1])
    return float(s)

### Fact check the total hours and ensure that they match the csv

In [75]:
# sum all time entries to ensure code is working correctly to compare with csv summed time entries 
python_total = compd["Hours"].sum()
python_total

1886.0

In [76]:
compd.groupby(["Phase", "Name"])["Hours"].sum()

Phase                              Name             
300FD Final Design (Architecture)  Bellamy, Adam        11.50
                                   Divis, Sarah          0.25
                                   Gosnell, Wyatt       91.00
                                   Henke, David          5.00
                                   Meyer, Jarred        54.00
                                                        ...  
604RP RPR Services (Construction)  Wilshusen, Andrew     1.00
803PC Post Construction (WIG)      Doane, Tyler          4.00
                                   Price, Amanda         0.25
                                   Schultes, Michael     1.00
                                   Wilkins, Devan        7.00
Name: Hours, Length: 72, dtype: float64

### Summarize compiled hours for each person

In [77]:
summary = (
    compd.groupby("Phase", as_index=False)["Hours"]
         .sum()
         .rename(columns={"Hours": "Total_Hours"})
         .sort_values("Total_Hours", ascending=False)
)

In [78]:
summ = (
    compd.groupby("Name", as_index=False)["Hours"]
        .sum()
)
summ

,Name,Hours
0,"Adkins, Justin",2.50
1,"Baldridge, Anthony",1.25
2,"Barker, Roxanne",5.00
3,"Bellamy, Adam",11.50
4,"Cerny, Caden",11.25
5,"Davis, Susan",77.50
6,"Divis, Sarah",0.75
7,"Doane, Tyler",577.00
8,"Dominguez, Brian",22.50
9,"Dryden, Noah",39.50


In [79]:
person_phase = (
    compd.groupby(["Phase", "Name"], as_index=False)["Hours"]
         .sum()
         .rename(columns={"Hours": "Total_Hours"})
         .sort_values(["Phase", "Total_Hours"], ascending=[True, False])
)
person_phase

,Phase,Name,Total_Hours
2,300FD Final Design (Architecture),"Gosnell, Wyatt",91.00
4,300FD Final Design (Architecture),"Meyer, Jarred",54.00
0,300FD Final Design (Architecture),"Bellamy, Adam",11.50
5,300FD Final Design (Architecture),"Roudebush, Blake",7.00
3,300FD Final Design (Architecture),"Henke, David",5.00
...,...,...,...
65,604RP RPR Services (Construction),"Roudebush, Blake",0.00
71,803PC Post Construction (WIG),"Wilkins, Devan",7.00
68,803PC Post Construction (WIG),"Doane, Tyler",4.00
70,803PC Post Construction (WIG),"Schultes, Michael",1.00


In [94]:
hours_by_phase = compd.groupby("Phase")["Hours"].sum()
hours_by_phase

Phase
300FD Final Design (Architecture)               168.75
302FD Final Design (Electrical)                 156.25
303FD Final Design (WIG)                        497.50
305FD Final Design (Survey)                       1.50
307FD Final Design (Transportation)              79.75
402BN Bidding and Negotiation (Electrical)        0.50
403BN Bidding and Negotiation (WIG)              41.50
500CS Construction Services (Architecture)        4.25
502CS Construction Services (Electrical)         10.00
503CS Construction Services (WIG)               288.25
505CS Construction Services (Survey)              6.00
507CS Construction Services (Transportation)     28.50
602RP RPR Services (Electrical)                   0.00
603RP RPR Services (WIG)                         43.00
604RP RPR Services (Construction)               548.00
803PC Post Construction (WIG)                    12.25
Name: Hours, dtype: float64

### Add in staff directory csv to indentify total hours per title

In [82]:
os.listdir("/home/9a8ae02a-2a7d-4aaa-a79e-235075eedfa6/Fee Chart Code")
path = "staffdierectory.csv"   # ← example, USE EXACT NAME YOU SEE

In [96]:
import pandas as pd

titles = pd.read_csv("staffdierectory.csv", encoding="cp1252")

In [97]:
titles.head()

,Name,Title,Notes,Unnamed: 3,Unnamed: 4
0,"Anderson, Seth",Electrical Dept,environmental,NaN,ElectricaL?? I'm not sure
1,"Arens, Steve",PE,structural pm,NaN,NaN
2,"Baldridge, Anthony",Electrical Dept,NaN,NaN,NaN
3,"Barker, Denise",Admin,NaN,NaN,NaN
4,"Bartja, Robert",IDS,"Visualization team, works on rendering, not in...",NaN,NaN


In [99]:
# Clean column names (just in case)
person_phase.columns = person_phase.columns.str.strip()
titles.columns = titles.columns.str.strip()

# Keep only needed columns from titles
titles_small = titles[["Name", "Title"]].copy()

# Clean join keys
person_phase["Name"] = person_phase["Name"].astype(str).str.strip()
titles_small["Name"] = titles_small["Name"].astype(str).str.strip()

# Merge
person_phase_with_title = person_phase.merge(
    titles_small,
    on="Name",
    how="left"
)

# Fill missing titles
person_phase_with_title["Title"] = person_phase_with_title["Title"].fillna("UNKNOWN / NOT FOUND")

# Total hours by Title
hours_by_title = (
    person_phase_with_title
        .groupby("Title", as_index=False)["Total_Hours"]
        .sum()
        .sort_values("Total_Hours", ascending=False)
)

hours_by_title

,Title,Total_Hours
7,PE,635.00
3,Construction,569.00
1,Architecture Dept,167.75
0,Admin,113.25
8,PM,104.25
2,CAD,103.00
9,Structural Dept,95.00
5,Electrical Dept,88.75
10,Survey Dept,6.00
6,LA,3.25


In [102]:
# sum all time entries to ensure code is working correctly to compare with csv summed time entries 
total_hours = person_phase_with_title["Total_Hours"].sum()
print("Total Hours:", total_hours)

Total Hours: 1886.0


In [100]:
unknown_people = person_phase_with_title[
    person_phase_with_title["Title"] == "UNKNOWN / NOT FOUND"
]

unknown_people.sort_values("Total_Hours", ascending=False)

,Phase,Name,Total_Hours,Title
